### CNN on CIFR Assignment:

1.  Please visit this link to access the state-of-art DenseNet code for reference - DenseNet - cifar10 notebook link
2.  You need to create a copy of this and "retrain" this model to achieve 90+ test accuracy. 
3.  You cannot use DropOut layers.
4.  You MUST use Image Augmentation Techniques.
5.  You cannot use an already trained model as a beginning points, you have to initilize as your own
6.  You cannot run the program for more than 300 Epochs, and it should be clear from your log, that you have only used 300 Epochs
7.  You cannot use test images for training the model.
8.  You cannot change the general architecture of DenseNet (which means you must use Dense Block, Transition and Output blocks as mentioned in the code)
9.  You are free to change Convolution types (e.g. from 3x3 normal convolution to Depthwise Separable, etc)
10. You cannot have more than 1 Million parameters in total
11. You are free to move the code from Keras to Tensorflow, Pytorch, MXNET etc. 
12. You can use any optimization algorithm you need. 
13. You can checkpoint your model and retrain the model from that checkpoint so that no need of training the model from first if you lost at any epoch while training. You can directly load that model and Train from that epoch. 

In [1]:
# import keras
# from keras.datasets import cifar10
# from keras.models import Model, Sequential
# from keras.layers import Dense, Dropout, Flatten, Input, AveragePooling2D, merge, Activation
# from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
# from keras.layers import Concatenate
# from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import models, layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import BatchNormalization, Activation, Flatten
from tensorflow.keras.optimizers import Adam

In [2]:
import tensorflow as tf
import numpy as np

In [3]:
# Hyperparameters
batch_size = 128
num_classes = 10
epochs = 90
l = 40
num_filter = 12
compression = 0.5
dropout_rate = 0.2

In [4]:
# Load CIFAR10 Data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
img_height, img_width, channel = X_train.shape[1],X_train.shape[2],X_train.shape[3]

# convert to one hot encoing 
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes) 

170500096/170498071 [==============================] - 11s 0us/step


In [5]:
# X_train = X_train.astype('float32')
# X_test = X_test.astype('float32')

# def preprocess_data(data_set):
#     mean = np.array([125.3, 123.0, 113.9])
#     std = np.array([63.0, 62.1, 66.7])

#     data_set -= mean
#     data_set /= std
#     return data_set

# X_train = preprocess_data(X_train)
# X_test = preprocess_data(X_test)
def normalize_pixels(train, test):
    '''
    Normalize data into range of 0 to 1
    '''
    train_norm = train.astype('float32')
    test_norm  = test.astype('float32')
    
    train_norm /= 255
    test_norm /= 255
    
    return (train_norm, test_norm)
X_train,X_test=normalize_pixels(X_train,X_test)    

In [6]:
datagen_train = ImageDataGenerator(
   width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True
)
# datagen_train = ImageDataGenerator(
#     rotation_range=20,
#     width_shift_range=0.125,
#     height_shift_range=0.125,
#     horizontal_flip=True,
#     fill_mode='nearest',
#     zoom_range=0.10
# )


datagen_train.fit(X_train)

Dense Block

In [7]:
def denseblock(input, num_filter = 12, dropout_rate = 0.2):
    global compression
    temp = input
    for _ in range(l): 
        BatchNorm = layers.BatchNormalization()(temp)
        relu = layers.Activation('relu')(BatchNorm)
        Conv2D_3_3 = layers.Conv2D(int(num_filter*compression), (3,3), use_bias=False ,padding='same')(relu)
        if dropout_rate>0:
            Conv2D_3_3 = layers.Dropout(dropout_rate)(Conv2D_3_3)
        concat = layers.Concatenate(axis=-1)([temp,Conv2D_3_3])
        
        temp = concat
        
    return temp

Transition

In [8]:
def transition(input, num_filter = 12, dropout_rate = 0.2):
    global compression
    BatchNorm = layers.BatchNormalization()(input)
    relu = layers.Activation('relu')(BatchNorm)
    Conv2D_BottleNeck = layers.Conv2D(int(num_filter*compression), (1,1), use_bias=False ,padding='same')(relu)
    if dropout_rate>0:
         Conv2D_BottleNeck = layers.Dropout(dropout_rate)(Conv2D_BottleNeck)
    avg = layers.AveragePooling2D(pool_size=(2,2))(Conv2D_BottleNeck)
    return avg



Output

In [9]:
def output_layer(input):
    global compression
    BatchNorm = layers.BatchNormalization()(input)
    relu = layers.Activation('relu')(BatchNorm)
    AvgPooling = layers.AveragePooling2D(pool_size=(2,2))(relu)
    flat = layers.Flatten()(AvgPooling)
    output = layers.Dense(num_classes, activation='softmax')(flat)
    return output

In [10]:
num_filter = 40
dropout_rate = 0
l = 12
input = layers.Input(shape=(img_height, img_width, channel,))
First_Conv2D = layers.Conv2D(num_filter, (3,3), use_bias=False ,padding='same')(input)

First_Block = denseblock(First_Conv2D, num_filter, dropout_rate)
First_Transition = transition(First_Block, num_filter, dropout_rate)

Second_Block = denseblock(First_Transition, num_filter, dropout_rate)
Second_Transition = transition(Second_Block, num_filter, dropout_rate)

Third_Block = denseblock(Second_Transition, num_filter, dropout_rate)
Third_Transition = transition(Third_Block, num_filter, dropout_rate)

Last_Block = denseblock(Third_Transition,  num_filter, dropout_rate)
output = output_layer(Last_Block)

In [11]:
model = Model(inputs=[input], outputs=[output])

In [12]:
model.compile(loss='categorical_crossentropy',
              optimizer="adam",
              metrics=['accuracy'])

In [13]:

history = model.fit_generator(
    datagen_train.flow(X_train, y_train, batch_size=128),
    steps_per_epoch=(len(X_train)/128),
    epochs=150,
    verbose = 1,
    validation_data=(X_test, y_test)
)

Instructions for updating:
Please use Model.fit, which supports generators.
Epoch 1/150
391/390 [==============================] - 217s 555ms/step - loss: 1.4519 - accuracy: 0.4678 - val_loss: 2.1097 - val_accuracy: 0.3279
Epoch 2/150
391/390 [==============================] - 211s 540ms/step - loss: 0.9864 - accuracy: 0.6455 - val_loss: 1.2030 - val_accuracy: 0.5763
Epoch 3/150
391/390 [==============================] - 212s 542ms/step - loss: 0.7914 - accuracy: 0.7214 - val_loss: 1.2236 - val_accuracy: 0.6321
Epoch 4/150
391/390 [==============================] - 212s 542ms/step - loss: 0.6765 - accuracy: 0.7637 - val_loss: 0.9502 - val_accuracy: 0.6849
Epoch 5/150
391/390 [==============================] - 212s 541ms/step - loss: 0.5985 - accuracy: 0.7916 - val_loss: 0.7863 - val_accuracy: 0.7355
Epoch 6/150
391/390 [==============================] - 212s 542ms/step - loss: 0.5538 - accuracy: 0.8093 - val_loss: 0.6863 - val_accuracy: 0.7671
Epoch 7/150
391/390 [=====================

KeyboardInterrupt: ignored

In [14]:
score = model.evaluate(X_test, y_test, verbose=1)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

313/313 [==============================] - 13s 41ms/step - loss: 0.5292 - accuracy: 0.8989
Test loss: 0.5292410254478455
Test accuracy: 0.8988999724388123
